In [1]:
!ls

avisadores_acusticos.csv
cabezas_semaforos.csv
cruces_semaforizados.csv
Estructura_DS_CrucesSemaforos.pdf
Estructura_DS_Datos_Abiertos_Cabezas_Semaforos.pdf
traffic_light_signals_acoustic.json
traffic_light_signals.json
Traffic_lights_Madrid.ipynb
Untitled.ipynb


In [2]:
!pip install folium

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
import pandas as pd
import random
import json
import folium

# cruces_semaforizados

This dataset has the following information about traffic lights (semáforos) in Madrid:
- distrito (neighborhood) where is installed
- description, in texte, is the cross of streets where it is installed
- longitud and latitud, which form the GPS coordination
- utm_x and utm_y that gives the position in this system
- fecha_inst which states the date when the light was installed
- id, unique identifier.

For our study we only need the id, description, and GPS position (lat, long).
Also, we need to add an status (red, yellow, green) to determine the current color of the traffic light. Once this created, we are not changing it and leave it static for the analysis. The probabilities of being RYG are: 0.45, .10, 0.45

In [4]:
# Load the CSV file (semicolon-separated)
df_cruces = pd.read_csv("cruces_semaforizados.csv", sep=';', quotechar='"', encoding='latin1') # or 'cp1252'
df_cruces.head(10)

,distrito,id,descripcion,fecha_inst,utm_x,utm_y,longitud,latitud
0,11,565,ANTONIO LEYVA - NAVAHONDA,23/04/1973,439293.957512,4.471615e+06,-3.715293,40.392922
1,8,2028,MONASTERIO SUSO Y YUSO 49 - P.P.,17/09/2007,441236.282378,4.484123e+06,-3.693566,40.505737
2,20,654,HERMANOS GARCIA NOBLEJAS - EMILIO MUÑOZ,14/05/1975,446364.834913,4.475691e+06,-3.632326,40.430129
3,16,4744,GLORIETA JOSE LUIS FERNANDEZ DEL AMO,21/05/2023,445556.378435,4.481742e+06,-3.642376,40.484588
4,16,4749,AV. FUERZAS ARMADAS 116 P.P.,21/05/2023,446427.157374,4.481892e+06,-3.632115,40.485992
5,4,138,GOYA - LAGASCA,21/06/1965,441858.379882,4.475185e+06,-3.685405,40.425268
6,19,1002,CALAHORRA - CALAHORRA,12/04/1981,448091.534392,4.473196e+06,-3.611767,40.407759
7,18,710,CRA. VILLAVERDE - CNO. POZO TIO RAIMUNDO - P.P.,20/09/1976,445193.837316,4.469002e+06,-3.645555,40.369798
8,18,399,PL. SIERRA GADOR - REAL ARGANDA,25/05/1970,447249.303635,4.470091e+06,-3.621435,40.379736
9,5,1330,MARCENADO - SAN ERNESTO,11/08/1992,442838.724987,4.477847e+06,-3.674088,40.449317


### Adding the light status

R -> 45%
Y -> 10%
G -> 45%

In [5]:
# Possible values
values = ["red", "orange", "green"]

# Corresponding probabilities
probs = [0.45, 0.10, 0.45]

# Add the column
np.random.seed(12)
df_cruces["status"] = np.random.choice(values, size=len(df_cruces), p=probs)

df_cruces.head()

,distrito,id,descripcion,fecha_inst,utm_x,utm_y,longitud,latitud,status
0,11,565,ANTONIO LEYVA - NAVAHONDA,23/04/1973,439293.957512,4.471615e+06,-3.715293,40.392922,red
1,8,2028,MONASTERIO SUSO Y YUSO 49 - P.P.,17/09/2007,441236.282378,4.484123e+06,-3.693566,40.505737,green
2,20,654,HERMANOS GARCIA NOBLEJAS - EMILIO MUÑOZ,14/05/1975,446364.834913,4.475691e+06,-3.632326,40.430129,red
3,16,4744,GLORIETA JOSE LUIS FERNANDEZ DEL AMO,21/05/2023,445556.378435,4.481742e+06,-3.642376,40.484588,orange
4,16,4749,AV. FUERZAS ARMADAS 116 P.P.,21/05/2023,446427.157374,4.481892e+06,-3.632115,40.485992,red


# cabezas_semáforos

This dataset describes the information related to light head's installed in Madrid.

In [6]:
# Load the CSV file (semicolon-separated)
df_cabezas = pd.read_csv("cabezas_semaforos.csv", sep=';', quotechar='"', encoding='latin1')   # or 'cp1252'
df_cabezas.head(10)

,tipo_elem,distrito,id,id_cruce,fecha_inst,utm_x,utm_y,longitud,latitud
0,other,10,61195,895,2015-07-13 00:00:00,437433.498707,4.472043e+06,-3.737255,40.396645
1,other,10,61194,895,2015-07-13 00:00:00,437440.723422,4.472040e+06,-3.737169,40.396612
2,other,10,61197,895,2025-06-03 00:00:00,437438.580002,4.472044e+06,-3.737195,40.396652
3,other,10,61196,895,2025-06-03 00:00:00,437434.296358,4.472048e+06,-3.737246,40.396683
4,other,10,61198,895,2025-06-03 00:00:00,437441.648752,4.472044e+06,-3.737159,40.396651
5,other,10,4676,895,2015-07-13 00:00:00,437409.330174,4.472019e+06,-3.737537,40.396427
6,other,3,49919,2159,2013-06-24 00:00:00,443077.550462,4.472139e+06,-3.670761,40.397914
7,other,19,60804,4810,2025-03-28 17:47:00,452705.845316,4.472704e+06,-3.557351,40.403607
8,other,19,60805,4810,2025-03-28 17:47:00,452697.583244,4.472693e+06,-3.557448,40.403503
9,other,19,60826,4811,2025-03-28 17:47:00,453079.655730,4.473302e+06,-3.552990,40.409014


# avisadores_acústicos

This dataset describes the traffic lights that have a sound notification for people with sight impairments. 

In [7]:
# Load the CSV file (semicolon-separated)
df_avisadores = pd.read_csv("avisadores_acusticos.csv", sep=';', quotechar='"', encoding='latin1')   # or 'cp1252'
df_avisadores.head(10)

,tipo_elem,distrito,id,id_cruce,localizaci,num,fecha_inst,utm_x,utm_y,longitud,latitud
0,AVISADOR ACUSTICO SIN POSIBILIDAD DE CONFIGURA...,1,3442,99,CALLE SAN BERNARDO,94.0,2018-08-07 00:00:00,440156.139765,4.475728e+06,-3.705522,40.430040
1,AVISADOR ACUSTICO SIN POSIBILIDAD DE CONFIGURA...,1,3443,99,GTA. RUIZ JIMENEZ,24.0,2018-08-07 00:00:00,440178.420911,4.475682e+06,-3.705254,40.429622
2,AVISADOR ACUSTICO SIN POSIBILIDAD DE CONFIGURA...,1,3446,99,CALLE SAN BERNARDO,91.0,2018-08-07 00:00:00,440112.340111,4.475633e+06,-3.706029,40.429175
3,AVISADOR ACUSTICO SIN POSIBILIDAD DE CONFIGURA...,1,3439,99,GTA. RUIZ JIMENEZ,1.0,2018-08-07 00:00:00,440085.398333,4.475673e+06,-3.706350,40.429541
4,AVISADOR ACUSTICO SIN POSIBILIDAD DE CONFIGURA...,1,3440,99,CALLE ALBERTO AGUILERA,2.0,2018-08-07 00:00:00,440083.579049,4.475698e+06,-3.706374,40.429764
5,AVISADOR ACUSTICO SIN POSIBILIDAD DE CONFIGURA...,1,3441,99,GTA. RUIZ JIMENEZ,93.0,2018-08-07 00:00:00,440136.118329,4.475729e+06,-3.705758,40.430045
6,AVISADOR ACUSTICO SIN POSIBILIDAD DE CONFIGURA...,1,3444,99,GTA. RUIZ JIMENEZ,6.0,2018-08-07 00:00:00,440178.064836,4.475658e+06,-3.705256,40.429406
7,AVISADOR ACUSTICO SIN POSIBILIDAD DE CONFIGURA...,1,3445,99,CALLE SAN BERNARDO,92.0,2018-08-07 00:00:00,440128.127262,4.475634e+06,-3.705843,40.429186
8,AVISADOR ACUSTICO SIN POSIBILIDAD DE CONFIGURA...,1,4396,2096,PLAZA SANTA BA?RBARA,1.0,2018-08-01 00:00:00,440891.636865,4.475345e+06,-3.696815,40.426640
9,AVISADOR ACUSTICO SIN POSIBILIDAD DE CONFIGURA...,1,4398,2096,PLAZA SANTA BA?RBARA,1.0,2018-08-01 00:00:00,440887.225742,4.475345e+06,-3.696867,40.426644


In [8]:
len(df_cabezas)

57625

In [9]:
len(df_cruces)

2489

In [10]:
len(df_avisadores)

10635

In [11]:
# Create a base map centered over Madrid
m = folium.Map(location=[40.42, -3.70], zoom_start=12, tiles="OpenStreetMap")

# Add the points to the map
for _, row in df_cruces[df_cruces['distrito']==3].iterrows():
    popup_text = f"""
    <b>{row['descripcion']}</b><br>
    Distrito: {row['distrito']}<br>
    Fecha instalación: {row['fecha_inst']}<br>
    Lon: {row['longitud']:.6f}, Lat: {row['latitud']:.6f}
    """
    folium.Marker(
        location=[row['latitud'], row['longitud']],
        popup=popup_text,
        tooltip=row['descripcion'],
        icon=folium.Icon(color=row['status'], icon="traffic-light", prefix='fa')
    ).add_to(m)

In [12]:
# Display the map
m

In [13]:
# Create a base map centered over Madrid
m = folium.Map(location=[40.42, -3.70], zoom_start=12, tiles="OpenStreetMap")

# Add the points to the map
for _, row in df_cabezas[df_cabezas['distrito']==3].iterrows():
    popup_text = f"""
    <b>{row['id_cruce']}</b><br>
    Distrito: {row['distrito']}<br>
    Fecha instalación: {row['fecha_inst']}<br>
    Lon: {row['longitud']:.6f}, Lat: {row['latitud']:.6f}
    """
    folium.Marker(
        location=[row['latitud'], row['longitud']],
        popup=popup_text,
        tooltip=row['id_cruce'],
        icon=folium.Icon(color="red", icon="traffic-light", prefix='fa')
    ).add_to(m)

In [14]:
m

# Convertion of CSV into JSON -> NGSI-LD Entities

## Traffic lights signal (is the intersection/junction of two streets)

In [15]:

# Build NGSI-LD JSON entities
entities = []
for _, row in df_cruces.iterrows():
    entity = {
        "id": f"urn:ngsi-ld:TrafficLightSignal:{row['id']}",
        "type": "TrafficLightSignal",
        "district": {
            "type": "Property",
            "value": int(row["distrito"])
        },
        "description": {
            "type": "Property",
            "value": row["descripcion"]
        },
        "installationDate": {
            "type": "Property",
            "value": row["fecha_inst"]
        },
        "status": {
            "type": "VocabProperty",
            "value": "yellow" if "orange" == row["status"] else row["status"]
            
        },
        "location": {
            "type": "GeoProperty",
            "value": {
                "type": "Point",
                "coordinates": [float(row["longitud"]), float(row["latitud"])]
            }
        }
    }
    entities.append(entity)

# Convert to JSON array (pretty-printed)
json_output = json.dumps(entities, indent=2, ensure_ascii=False)
with open("traffic_light_signals.json", "w", encoding="utf-8") as f:
    f.write(json_output)
print("File written")

File written


## Traffic lights acoustic signal

In [39]:
# Convert to NGSI-LD JSON entities
entities_acoustic = []
for _, row in df_avisadores.iterrows():
    entity = {
        "id": f"urn:ngsi-ld:TrafficLightSignalAcoustic:{int(row['id'])}",
        "type": "TrafficLightSignalAcoustic",
        "elementType": {
            "type": "Property",
            "value": row["tipo_elem"]
        },
        "district": {
            "type": "Property",
            "value": int(row["distrito"])
        },
        "locationName": {
            "type": "Property",
            "value": row["localizaci"]
        },
        "number": {
            "type": "Property",
            "value": row["num"]
        },
        "installationDate": {
            "type": "Property",
            "value": str(row["fecha_inst"])
        },
        "refTrafficLightSignal": {
            "type": "Relationship",
            "object": f"urn:ngsi-ld:TrafficLightSignal:{int(row['id_cruce'])}"
        },
        "location": {
            "type": "GeoProperty",
            "value": {
                "type": "Point",
                "coordinates": [float(row["longitud"]), float(row["latitud"])]
            }
        }
    }
    entities_acoustic.append(entity)

# Export as JSON array (pretty format)
json_output = json.dumps(entities_acoustic, indent=2, ensure_ascii=False)
print(json_output)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [40]:
with open("traffic_light_signals_acoustic.json", "w", encoding="utf-8") as f:
    f.write(json_output)